In [ ]:
import sys
sys.path.append('/Users/joachim/texjs/lva/IntroSC/ASC-ODE/build/mechsystem')
sys.path.append('../build/mechsystem/Debug/')

from mass_spring import *

from pythreejs import *

In [ ]:
mss = MassSpringSystem3d()
mss.gravity = (0,0,-9.81)

m1 = mss.add (Mass(1, (1,0,0)))
m2 = mss.add (Mass(1, (2,0,0)))
f = mss.add (Fix( (0,0,0)) )
mss.add (DistanceConstraint(1, (f, m1)))
mss.add (DistanceConstraint(1, (m1, m2)))


In [ ]:
masses = []
for m in mss.masses:
    masses.append(
        Mesh(SphereBufferGeometry(0.2, 16, 16),
             MeshStandardMaterial(color='red'),
             position=m.pos)) 

fixes = []
for f in mss.fixes:
    fixes.append(
        Mesh(SphereBufferGeometry(0.2, 32, 16),
             MeshStandardMaterial(color='blue'),
             position=f.pos)) 

def connector_pos(con):
    if con.type == 1:   # FIX
        return mss.fixes[con.nr].pos
    else:               # MASS
        return mss.masses[con.nr].pos

springpos = []
for s in mss.springs:
    pA = connector_pos(s.connectors[0])
    pB = connector_pos(s.connectors[1])
    springpos.append([pA, pB])

springs = None
if springpos:
    springgeo = LineSegmentsGeometry(positions=springpos)
    springmat = LineMaterial(linewidth=3, color='cyan')
    springs = LineSegments2(springgeo, springmat)
     
constraintpos = []
for d in mss.distanceConstraints:
    pA = connector_pos(d.connectors[0])
    pB = connector_pos(d.connectors[1])
    constraintpos.append([pA, pB])

constraints = None
if constraintpos:
    constraintgeo = LineSegmentsGeometry(positions=constraintpos)
    constraintmat = LineMaterial(linewidth=2, color='green')
    constraints = LineSegments2(constraintgeo, constraintmat)


axes = AxesHelper(1)

In [ ]:
view_width = 600
view_height = 400

camera = PerspectiveCamera(position=[10, 6, 10], aspect=view_width/view_height)
key_light = DirectionalLight(position=[0, 10, 10])
ambient_light = AmbientLight()

scene_objects = [*masses, *fixes, axes, camera, key_light, ambient_light]

if springs is not None:
    scene_objects.append(springs)

if constraints is not None:
    scene_objects.append(constraints)

scene = Scene(children=scene_objects)

controller = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controller],
                    width=view_width, height=view_height)

renderer


In [ ]:
from time import sleep
for i in range(1000):
    mss.simulate (0.1, 500)
    for m,mvis in zip(mss.masses, masses):
        mvis.position = (m.pos[0], m.pos[1], m.pos[2])

    springpos = []
    for s in mss.springs:
        pA = mss[s.connectors[0]].pos
        pB = mss[s.connectors[1]].pos
        springpos.append ([ pA, pB ]) 
    springs.geometry = LineSegmentsGeometry(positions=springpos)
    sleep(0.01)